In [5]:
%pip install pandas numpy scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB ? eta 0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.1/9.9 MB 22.9 MB/s eta 0:00:01
   ---------- ----------------------------- 2.5/9.9 MB 26.6 MB/s eta 0:00:01
   --------------- ------------------------ 3.8/9.9 MB 27.0 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.9 MB 29.2 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 31.1 MB/s eta 0:00:01
   ----------------------------------- ---- 8.8/9.9 MB 31.3 MB/s eta 0:00:01
   ---------------------------------------  9.9/9.9 MB 31.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 30.1 MB/s eta 0:00:00
   -----------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# a) Read in the dataset
df = pd.read_csv("data_01.csv")

# read in the 5 first rows
print("a) Datasetet inläst. Här är de 5 första raderna:")
display(df.head())

a) Datasetet inläst. Här är de 5 första raderna:


,x1,x2,x3,x4,x5,target
0,0.743487,1.072825,1.332911,-1.244771,0.344978,220.173943
1,0.835264,0.202184,0.966480,0.745883,-0.033773,175.873929
2,-1.103234,0.030615,-0.140385,0.727683,-2.831224,-162.270054
3,1.210186,1.685258,-0.394123,0.719024,-2.166585,165.930461
4,0.474577,0.647737,-0.451812,-0.409472,-0.051473,43.250511


In [8]:
# b) split the data into X and y
target_col = 'target'
X = df.drop(target_col, axis=1)
y = df[target_col]
print(f"\nb) Uppdelning klar. '{target_col}' är satt som y.")


b) Uppdelning klar. 'target' är satt som y.


In [9]:
# c) split the data
# 1. 20% to test data. Remaining is 'X_train_full' (80% of total).
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 15% to the VALIDATION set from the REMAINING data (X_train_full).
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

print(f"\nc) Data uppdelad:")
print(f"   Träningsdata:   {X_train.shape[0]} rader")
print(f"   Valideringsdata:{X_val.shape[0]} rader")
print(f"   Testdata:       {X_test.shape[0]} rader")


c) Data uppdelad:
   Träningsdata:   135 rader
   Valideringsdata:24 rader
   Testdata:       40 rader


In [ ]:
# d) Train two models on the training data
lin_reg = LinearRegression()
tree_reg = DecisionTreeRegressor(random_state=42)

lin_reg.fit(X_train, y_train)
tree_reg.fit(X_train, y_train)
print("\nd) Modeller tränade (LinearRegression & DecisionTreeRegressor).")


d) Modeller tränade (LinearRegression & DecisionTreeRegressor).


In [ ]:
# e) Prediction on validation data
pred_lin = lin_reg.predict(X_val)
pred_tree = tree_reg.predict(X_val)


# we use sklearn.metrics.mean_squared_error and take sqrt for RMSE
rmse_lin = np.sqrt(mean_squared_error(y_val, pred_lin))
rmse_tree = np.sqrt(mean_squared_error(y_val, pred_tree))

print(f"\ne) Resultat på Valideringsdata (RMSE):")
print(f"   Linear Regression: {rmse_lin:.4f}")
print(f"   Decision Tree:     {rmse_tree:.4f}")

# choose the best model
if rmse_lin < rmse_tree:
    best_model = lin_reg
    model_name = "Linear Regression"
else:
    best_model = tree_reg
    model_name = "Decision Tree"

print(f"   -> Bäst modell var: {model_name}")


e) Resultat på Valideringsdata (RMSE):
   Linear Regression: 3.5923
   Decision Tree:     99.3594
   -> Bäst modell var: Linear Regression


In [ ]:
# f) train the best model on the combined training + validation data
best_model.fit(X_train_full, y_train_full)
print(f"\nf) {model_name} har tränats om på tränings- och valideringsdatan ihop.")


f) Linear Regression har tränats om på tränings- och valideringsdatan ihop.


In [ ]:
# g) evaluate the model on the test data
final_pred = best_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
print(f"g) Slutgiltig RMSE på testdata: {final_rmse:.4f}")

g) Slutgiltig RMSE på testdata: 3.3715


In [ ]:
# h) train the best model on the entire dataset
production_model = best_model
production_model.fit(X, y)
print(f"h) Produktionsmodellen är nu tränad på 100% av datasetet (alla {len(df)} rader).")

h) Produktionsmodellen är nu tränad på 100% av datasetet (alla 199 rader).
